In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('powerplant_data.csv')

In [3]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [4]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [5]:
df.shape

(9568, 5)

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [7]:
X= df.drop(columns='PE', axis=1)
y = df['PE']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
import torch
import torch.nn as nn

In [11]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [12]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [14]:
# Define Deep Learning 

class ANN(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = torch.nn.Sequential(
            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2nd Hidden Layer
            nn.Linear(6, 6),
            nn.ReLU(),

            # Output layer
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)
    

In [15]:
model = ANN()

In [16]:
import torch.optim as optim

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [17]:
epochs = 100

best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        output = model(xb)
        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    epoch_train_loss = train_loss / len(train_loader)

    # validation
    model.eval()
    test_loss = 0.0

    for xb, yb in test_loader:
        with torch.no_grad():
            output = model(xb)
            loss = criterion(output, yb)

            test_loss += loss.item()

    epoch_test_loss = test_loss / len(test_loader)

    print(f"EPOCH: {epoch+1} / {epochs} train_loss: {epoch_train_loss} & test_loss: {epoch_test_loss}")

    #Save_model:
    if epoch_test_loss < best_val_loss:
        best_val_loss = epoch_test_loss
        torch.save(model.state_dict(), 'best_model.pt')

EPOCH: 1 / 100 train_loss: 205830.13919270833 & test_loss: 204344.81744791666
EPOCH: 2 / 100 train_loss: 198680.55872395833 & test_loss: 188384.47239583332
EPOCH: 3 / 100 train_loss: 169292.21809895834 & test_loss: 145846.35859375
EPOCH: 4 / 100 train_loss: 118196.50677083334 & test_loss: 90602.141015625
EPOCH: 5 / 100 train_loss: 67894.1701904297 & test_loss: 49431.60553385417
EPOCH: 6 / 100 train_loss: 37775.85051269531 & test_loss: 28857.87958984375
EPOCH: 7 / 100 train_loss: 23431.595373535158 & test_loss: 19110.919189453125
EPOCH: 8 / 100 train_loss: 16494.448787434896 & test_loss: 14212.036376953125
EPOCH: 9 / 100 train_loss: 12700.475061035157 & test_loss: 11104.368701171876
EPOCH: 10 / 100 train_loss: 9871.75303548177 & test_loss: 8558.951928710938
EPOCH: 11 / 100 train_loss: 7494.891542561849 & test_loss: 6361.725512695312
EPOCH: 12 / 100 train_loss: 5460.920066324869 & test_loss: 4549.884065755208
EPOCH: 13 / 100 train_loss: 3863.8358301798503 & test_loss: 3202.636328125
EPOC

In [18]:
model.load_state_dict(torch.load('best_model.pt'))

<All keys matched successfully>

In [19]:
model.eval()
with torch.no_grad():
    y_train_pred = model(X_train_tensor)
    y_test_pred = model(X_test_tensor)

    train_mse_loss = criterion(y_train_pred, y_train_tensor)
    test_mse_loss = criterion(y_test_pred, y_test_tensor)

print("train_mse_loss: ", train_mse_loss.item())
print("test_mse_loss: ", test_mse_loss.item())

train_mse_loss:  20.757932662963867
test_mse_loss:  19.092588424682617


In [20]:
from sklearn.metrics import r2_score
score = r2_score(y_test, y_test_pred)
print(score)

0.9332763291107368


In [21]:
pred_df = pd.DataFrame(y_test_pred.numpy(), columns=['Predected'])
actual_df = pd.DataFrame(y_test.values, columns=["Actual"])

pd.concat([pred_df, actual_df], axis=1)

,Predected,Actual
0,435.107086,433.27
1,437.107025,438.16
2,461.125122,458.42
3,476.195221,480.82
4,435.158844,441.41
...,...,...
1909,451.368134,456.70
1910,431.671356,438.04
1911,467.342072,467.80
1912,431.152313,437.14
